# Sigiriya SinLlama evidence handoff

Start Neo4j Desktop on Windows and run the local evidence export from the Sigiriya project directory.

```powershell
.\.venv\Scripts\python.exe -B -X utf8 -m scripts.sinllama_handoff
```

In Colab, select **Runtime > Change runtime type > T4 GPU**. Run these cells in order in a fresh runtime. Upload the evidence JSON and runtime ZIP from `results/generation/`. Neo4j and E5 stay on Windows. The ZIP contains only the generation code and this handoff utility.

This checkpoint is a [Sinhala base completion model](https://huggingface.co/SAWithanage/SinLlama-Llama-3-8B-Merged). It uses a `Question:` / `Answer:` completion prompt. Accepted output must preserve the selected source sentences in order. Unverified text is kept for inspection, and the final answer uses the local fallback.


## 1. Check the GPU and install inference dependencies

In [1]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a T4 GPU runtime before continuing.")
print(torch.cuda.get_device_name(0), torch.__version__)


Tesla T4 2.14.1+cu130


In [2]:
%pip install -q transformers==4.57.1 accelerate==1.10.1 bitsandbytes==0.48.2


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [1]:
import torch
import transformers
import accelerate
import bitsandbytes
import huggingface_hub

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
Transformers: 4.57.1
Accelerate: 1.10.1
BitsAndBytes: 0.48.2
Hugging Face Hub: 0.36.2


## 2. Upload the exported evidence and runtime ZIP

In [2]:
import json
import sys
from pathlib import Path
from zipfile import ZipFile

from google.colab import files

uploaded = files.upload()
evidence_name = "sigiriya_sinllama_evidence.json"
runtime_name = "sigiriya_sinllama_runtime.zip"
if not {evidence_name, runtime_name} <= uploaded.keys():
    raise ValueError("Upload both files produced by the local exporter.")

runtime = Path("/content/sigiriya_sinllama_runtime")
expected = {
    "src/__init__.py", "src/config.py", "src/generation.py", "src/prompting.py",
    "scripts/sinllama_handoff.py",
}
with ZipFile(runtime_name) as archive:
    if set(archive.namelist()) != expected:
        raise ValueError("Unexpected runtime files. Re-export from the local project.")
    archive.extractall(runtime, members=sorted(expected))
sys.path.insert(0, str(runtime))
payload = json.loads(Path(evidence_name).read_text(encoding="utf-8-sig"))
print(f"Loaded {len(payload['questions'])} questions.")


Saving sigiriya_sinllama_evidence.json to sigiriya_sinllama_evidence.json
Saving sigiriya_sinllama_runtime.zip to sigiriya_sinllama_runtime.zip
Loaded 9 questions.


## 3. Generate and verify

The first eligible question downloads the merged weights into Colab and loads them with 4-bit NF4, double quantization, and FP16 compute. The original checkpoint files need Colab disk space even though GPU inference is quantized. One question runs at a time. If the runtime reports a generation error, review the fallback status and retry in a fresh T4 runtime once the cause is resolved.


In [3]:
from collections import Counter
from importlib.metadata import version

from scripts.sinllama_handoff import generate_batch, write_json
from src.generation import SinLlamaGenerator

generator = SinLlamaGenerator(local_files_only=False)
output = generate_batch(payload, generator)
output["runtime"] = {
    "gpu": torch.cuda.get_device_name(0),
    "versions": {pkg: version(pkg) for pkg in ("torch", "transformers", "accelerate", "bitsandbytes")},
}
result_path = Path("sigiriya_sinllama_results.json")
write_json(result_path, output)
print(Counter(row["generation_status"] for row in output["results"]))


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/331 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/19.3M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/737 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

... (10 outputs left out)

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/170 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:464: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Counter({'EXTRACTIVE_FALLBACK_NOT_VERIFIED': 7, 'GENERATION_SKIPPED': 2})


## 4. Review and download the results

In [4]:
for row in output["results"]:
    print(row["id"], row["generation_status"], row["fallback_reason"] or "")
    print(row["final_answer"], end="\n\n")

files.download(str(result_path))


factual EXTRACTIVE_FALLBACK_NOT_VERIFIED repetition_or_bad_format
මෙම සිතුවම්වල කාන්තා රූප දැකගත හැකි අතර ඒවා අප්සරාවන් හෝ දිව්‍ය කාන්තාවන් ලෙස අර්ථකථනය කර ඇත.

overview EXTRACTIVE_FALLBACK_NOT_VERIFIED repetition_or_bad_format
දැනට ඉතිරිව ඇති බිතුසිතුවම් භූමියෙන් මීටර් 100කට වඩා ඉහළින් පිහිටි පර්වතයේ කොටසක දැකගත හැකිය. ඒවා වෙත ළඟාවීම සඳහා සර්පිලාකාර පඩිපෙළක් භාවිත කරයි. මෙම සිතුවම්වල කාන්තා රූප දැකගත හැකි අතර ඒවා අප්සරාවන් හෝ දිව්‍ය කාන්තාවන් ලෙස අර්ථකථනය කර ඇත. අද ඉතිරිව ඇති සිතුවම් පැරණි කාලයේ පර්වතයේ බටහිර මුහුණත පුරා පැවති විශාල සිතුවම් සමූහයක කුඩා කොටසක් ලෙස සැලකේ. සීගිරි බිතුසිතුවම් ශ්‍රී ලංකාවේ පැරණි චිත්‍ර කලාව අධ්‍යයනය කිරීමේදී ඉතා වැදගත් වේ.

multi_topic EXTRACTIVE_FALLBACK_NOT_VERIFIED repetition_or_bad_format
සීගිරි බිතුසිතුවම්: දැනට ඉතිරිව ඇති බිතුසිතුවම් භූමියෙන් මීටර් 100කට වඩා ඉහළින් පිහිටි පර්වතයේ කොටසක දැකගත හැකිය. ඒවා වෙත ළඟාවීම සඳහා සර්පිලාකාර පඩිපෙළක් භාවිත කරයි. මෙම සිතුවම්වල කාන්තා රූප දැකගත හැකි අතර ඒවා අප්සරාවන් හෝ දිව්‍ය කාන්තාවන් ලෙස අර්ථකථනය කර ඇත. අද ඉතිරි

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Save the downloaded JSON under the local `results/generation/` directory. Each question retains the raw answer proposal after completion-marker cleanup, the verification result, selected sources, final answer, and any fallback reason. Review correctness and Sinhala phrasing separately from the exact-quote check.